# Simple: Phone Field Analysis with PAMOLA.CORE

**Goal**: Learn phone field analysis basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Analyze phone fields using execute()
- Extract country and operator codes
- Detect messenger mentions in comments

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/
        └── 01_phone_analyzer_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.phone import PhoneOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load data from `examples/data_examples/sample.csv`
- Auto-creates sample data if file missing
- Review dataset structure before proceeding

**What you'll see:**
- Dataset summary (50 records, 3 columns)
- First 10 records with phone numbers
- Column statistics showing unique values
- Phone formats: international (+country-area-number) with optional messenger mentions

**Note:** Sample includes various country codes and messenger mentions (telegram, whatsapp, viber)

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with phone numbers in various formats
    sample_data = pd.DataFrame({
        'record_id': range(1, 51),
        'phone': [
            '+1-555-0100',
            '+44-20-7946-0958',
            '+33-6-12-34-56-78',
            '+7-950-123-45-67, telegram',
            '+49-30-12345678',
            '+86-10-12345678',
            '+91-22-12345678',
            '+55-11-98765-4321',
            '+61-2-9876-5432',
            '+81-3-1234-5678',
            '+7-495-123-45-67, viber',
            '+1-212-555-0199',
            '+44-7911-123456, whatsapp',
            '+33-1-23-45-67-89',
            '+49-151-12345678',
            '+86-138-1234-5678',
            '+91-9876543210',
            '+55-21-3456-7890',
            '+61-412-345-678',
            '+81-90-1234-5678',
            '+7-926-123-45-67',
            '+1-415-555-0132, signal',
            '+44-131-496-0000',
            '+33-6-98-76-54-32, telegram',
            '+49-89-12345678',
            '+86-21-12345678',
            '+91-11-12345678',
            '+55-85-98765-4321',
            '+61-3-9876-5432',
            '+81-6-1234-5678',
            '+7-812-123-45-67, whatsapp',
            '+1-650-555-0178',
            '+44-20-3456-7890',
            '+33-4-12-34-56-78',
            '+49-221-12345678',
            '+86-755-12345678',
            '+91-40-12345678',
            '+55-31-3456-7890',
            '+61-7-3456-7890',
            '+81-52-123-4567',
            '+7-916-123-45-67',
            '+1-310-555-0144',
            '+44-7700-900123',
            '+33-6-11-22-33-44, viber',
            '+49-40-12345678',
            '+86-20-12345678',
            '+91-80-12345678',
            '+55-71-98765-4321',
            '+61-8-8765-4321',
            '+81-92-123-4567, line'
        ],
        'customer_name': [f'Customer {i}' for i in range(1, 51)],
        'contact_type': np.random.choice(['Personal', 'Business', 'Support'], 50)
    })
    
    # Add some null values
    sample_data.loc[np.random.choice(50, 3, replace=False), 'phone'] = np.nan
    
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 10 records:")
print(df.head(10))

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:20s} ({dtype_str:10s})")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE field_name** in `create_config_kwargs()`
   - Default: `"phone"`
   - Change to YOUR dataset's phone column name
2. Run to validate field and setup environment

**What you'll see:**
- Field validation (type, non-null count, sample values)
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and progress tracker ready (✓)

**Note:** Field must exist in dataset and contain phone number strings

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "phone"  # Field to analyze - CUSTOMIZE THIS!
    }

kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

print(f"✓ Field to process: '{field_name}'")
print(f"  Data type: {df[field_name].dtype}")
print(f"  Non-null count: {df[field_name].notna().sum()}")
print(f"  Sample values:")
for val in df[field_name].dropna().head(3):
    print(f"    {val}")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="phone_analysis",
    description="Phone field analysis",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Processing {field_name} field",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review configuration parameters below
- Run to execute phone field analysis
- Monitor progress bar (6 steps: parse → validate → extract → analyze → save → complete)

**Key parameters:**
- `field_name`: Field to analyze (phone string required)
- `min_frequency=1`: Include all values in dictionaries
- `patterns_csv=None`: Use default messenger patterns
- `country_codes=None`: Analyze all countries found
- `generate_visualization=True`: Creates distribution charts

**What you'll see:**
- Configuration summary with all parameters
- Progress bar: parse → normalize → extract codes → detect messengers → save → complete
- Completion status: "completed" (verify no errors)
- Artifact count (6-8 files expected)

**Note:** Analysis is non-destructive, extracts metadata without modifying original phones

In [ ]:
# Create operation with production-style configuration
operation = PhoneOperation(
    # Core parameters
    field_name=field_name,
    min_frequency=1,
    patterns_csv=None,  # Use default patterns
    country_codes=None,  # Analyze all countries
    
    # Output configuration
    output_format='json',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    chunk_size=10000,
    use_dask=False,
    npartitions=None,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Field:            {operation.field_name}")
print(f"  Min frequency:    {operation.min_frequency}")
print(f"  Patterns CSV:     {operation.patterns_csv or 'default'}")
print(f"  Country codes:    {operation.country_codes or 'all'}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing phone analysis...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load phone analysis from task_dir
- Review validation metrics and dictionaries
- Verify parsing success rates

**What you'll see:**
- Phone validation stats (valid count, format errors, normalization success)
- Country codes dictionary with distribution percentages
- Operator codes dictionary (top 10 most common)
- Messenger mentions dictionary (telegram, whatsapp, viber, etc.)
- Phone features (extension count, comment count)

**Note:** High normalization success rate (>95%) indicates clean phone data

In [ ]:
# ==========================================================
# 0️⃣ DISPLAY GENERATED ARTIFACTS
# ==========================================================
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# ==========================================================
# 1️⃣ LOAD OUTPUT FILES
# ==========================================================
output_dir = task_dir / "output"
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
    raise SystemExit

json_files = list(output_dir.glob("*.json"))
if not json_files:
    print("⚠️  No JSON output files found")
    raise SystemExit


def newest(files):
    return sorted(files, key=lambda f: f.stat().st_mtime, reverse=True)[0]


stats_files = [f for f in json_files if f.name.endswith("_stats.json")]
country_files = [f for f in json_files if "country_codes_dictionary" in f.name]
operator_files = [f for f in json_files if "operator_codes_dictionary" in f.name]
messenger_files = [f for f in json_files if "messenger_dictionary" in f.name]

stats_file = newest(stats_files) if stats_files else None
country_file = newest(country_files) if country_files else None
operator_file = newest(operator_files) if operator_files else None
messenger_file = newest(messenger_files) if messenger_files else None


# ==========================================================
# 2️⃣ PHONE STATS
# ==========================================================
if stats_file:
    with open(stats_file, "r", encoding="utf-8") as f:
        stats = json.load(f)

    mtime = datetime.fromtimestamp(stats_file.stat().st_mtime)

    print("\n" + "=" * 80)
    print("📊 PHONE ANALYSIS (STATS)")
    print("=" * 80)
    print(f"📄 File: {stats_file.name}")
    print(f"🕒 Modified: {mtime:%Y-%m-%d %H:%M:%S}")

    print("\n📈 BASIC COUNTS")
    print(f"  Total rows           : {stats.get('total_rows', 0):,}")
    print(f"  Valid phones         : {stats.get('valid_count', 0):,} ({stats.get('valid_percentage', 0):.2f}%)")
    print(f"  Null phones          : {stats.get('null_count', 0):,} ({stats.get('null_percentage', 0):.2f}%)")
    print(f"  Format errors        : {stats.get('format_error_count', 0):,}")

    print("\n🔄 NORMALIZATION")
    print(
        f"  Success              : {stats.get('normalization_success_count', 0):,} "
        f"({stats.get('normalization_success_percentage', 0):.2f}%)"
    )

    print("\n📎 PHONE FEATURES")
    print(f"  With extension       : {stats.get('has_extension_count', 0):,}")
    print(f"  With comments        : {stats.get('has_comment_count', 0):,}")

    if stats.get("format_error_examples"):
        print("\n⚠️  FORMAT ERROR EXAMPLES")
        for ex in stats["format_error_examples"][:5]:
            print(f"  - {ex}")


# ==========================================================
# 3️⃣ COUNTRY CODES DICTIONARY
# ==========================================================
if country_file:
    with open(country_file, "r", encoding="utf-8") as f:
        c = json.load(f)

    print("\n" + "=" * 80)
    print("🌍 COUNTRY CODES DICTIONARY")
    print("=" * 80)
    print(f"📄 File: {country_file.name}")

    for row in c.get("country_codes", []):
        print(
            f"  +{row['country_code']:<3} "
            f"{row['count']:>5,} phones "
            f"({row['percentage']:>6.2f}%)"
        )


# ==========================================================
# 4️⃣ OPERATOR CODES DICTIONARY
# ==========================================================
if operator_file:
    with open(operator_file, "r", encoding="utf-8") as f:
        o = json.load(f)

    print("\n" + "=" * 80)
    print("📡 OPERATOR CODES DICTIONARY (TOP 10)")
    print("=" * 80)
    print(f"📄 File: {operator_file.name}")

    for row in o.get("operator_codes", [])[:10]:
        print(
            f"  +{row['country_code']}-{row['operator_code']:<4} "
            f"{row['count']:>4,} "
            f"({row['percentage']:>5.2f}%)"
        )


# ==========================================================
# 5️⃣ MESSENGER DICTIONARY
# ==========================================================
if messenger_file:
    with open(messenger_file, "r", encoding="utf-8") as f:
        m = json.load(f)

    print("\n" + "=" * 80)
    print("💬 MESSENGER MENTIONS")
    print("=" * 80)
    print(f"📄 File: {messenger_file.name}")

    for row in m.get("messengers", []):
        print(
            f"  {row['messenger']:<10} "
            f"{row['count']:>4,} "
            f"({row['percentage']:>5.2f}%)"
        )


# ==========================================================
# 6️⃣ FINAL SUMMARY
# ==========================================================
print("\n" + "=" * 80)
print("✨ SUMMARY")
print("=" * 80)
print(f"  Dataset size        : {stats.get('total_rows', 0):,}")
print(f"  Valid phones        : {stats.get('valid_count', 0):,}")
print(f"  Unique countries    : {len(stats.get('country_codes', {})):,}")
print(f"  Unique operators    : {len(stats.get('operator_codes', {})):,}")
print("\n✅ Phone analysis complete!")


## Step 6: Review Artifacts Location

**How to use:**
- Run to display all generated files
- Navigate to directories for manual inspection
- Verify each folder has expected file count

**What you'll see:**
- Directory structure tree (output/, dictionaries/, visualizations/, config/)
- File count per directory (typically 2-3 files each)
- File names with sizes in KB
- Absolute path to task directory for manual navigation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Phone stats JSON
├── dictionaries/     # Country/operator/messenger CSVs and JSONs
├── visualizations/   # Distribution charts (PNG)
└── config/           # Operation configuration
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'dictionaries', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Statistical Analysis**: Phone validation and parsing metrics
2. **Dictionaries**: Country codes, operator codes, messenger mentions
3. **Visualizations**: Distribution charts

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. STATISTICAL ANALYSIS (NEWEST)
print("\n1️⃣ STATISTICAL ANALYSIS (JSON):")
print("-" * 80)

output_dir = task_dir / "output"
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    json_files = list(output_dir.glob("*.json"))
    if not json_files:
        print("⚠️  No JSON output files found")
    else:
        try:
            # -----------------------------------------------------
            # Group files by type
            # -----------------------------------------------------
            stats_files = [f for f in json_files if f.name.endswith("_stats.json")]
            country_files = [f for f in json_files if "country_codes_dictionary" in f.name]
            operator_files = [f for f in json_files if "operator_codes_dictionary" in f.name]
            messenger_files = [f for f in json_files if "messenger_dictionary" in f.name]

            def load_latest(files):
                if not files:
                    return None
                files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
                return files[0]

            stats_file = load_latest(stats_files)
            country_file = load_latest(country_files)
            operator_file = load_latest(operator_files)
            messenger_file = load_latest(messenger_files)

            # =====================================================
            # 1️⃣ PHONE STATS
            # =====================================================
            if stats_file:
                mtime = datetime.fromtimestamp(stats_file.stat().st_mtime)
                print("\n📊 PHONE STATISTICS")
                print("=" * 80)
                print(f"📄 File: {stats_file.name}")
                print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")

                with open(stats_file, "r", encoding="utf-8") as f:
                    s = json.load(f)

                print("🔢 COUNTS:")
                print(f"   Total rows                 : {s.get('total_rows', 0):,}")
                print(f"   Null phones                : {s.get('null_count', 0):,} ({s.get('null_percentage', 0):.2f}%)")
                print(f"   Non-null phones            : {s.get('non_null_count', 0):,}")
                print(f"   Valid phones               : {s.get('valid_count', 0):,} ({s.get('valid_percentage', 0):.2f}%)")
                print(f"   Format errors              : {s.get('format_error_count', 0):,}")
                print(f"   Normalization success      : {s.get('normalization_success_count', 0):,} ({s.get('normalization_success_percentage', 0):.2f}%)")

                print("\n📎 PHONE FEATURES:")
                print(f"   Has extension              : {s.get('has_extension_count', 0):,}")
                print(f"   Has messenger comment      : {s.get('has_comment_count', 0):,}")

                if s.get("format_error_examples"):
                    print("\n⚠️  FORMAT ERROR EXAMPLES:")
                    for ex in s["format_error_examples"][:5]:
                        print(f"   - {ex}")

            # =====================================================
            # 2️⃣ COUNTRY CODES DICTIONARY
            # =====================================================
            if country_file:
                mtime = datetime.fromtimestamp(country_file.stat().st_mtime)
                print("\n🌍 COUNTRY CODES DICTIONARY")
                print("=" * 80)
                print(f"📄 File: {country_file.name}")
                print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")

                with open(country_file, "r", encoding="utf-8") as f:
                    c = json.load(f)

                print(f"   Total phones        : {c.get('total_phones', 0):,}")
                print(f"   Unique country codes: {c.get('total_country_codes', 0)}\n")

                for row in c.get("country_codes", []):
                    print(
                        f"   +{row['country_code']:<4} "
                        f": {row['count']:>5,} phones "
                        f"({row['percentage']:>6.2f}%)"
                    )

            # =====================================================
            # 3️⃣ OPERATOR CODES DICTIONARY
            # =====================================================
            if operator_file:
                mtime = datetime.fromtimestamp(operator_file.stat().st_mtime)
                print("\n📡 OPERATOR CODES DICTIONARY")
                print("=" * 80)
                print(f"📄 File: {operator_file.name}")
                print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")

                with open(operator_file, "r", encoding="utf-8") as f:
                    o = json.load(f)

                print(f"   Total phones        : {o.get('total_phones', 0):,}")
                print(f"   Unique operators    : {o.get('total_operator_codes', 0)}\n")

                print("   Top operator codes:")
                for row in o.get("operator_codes", [])[:10]:
                    print(
                        f"   +{row['country_code']}-{row['operator_code']:<4} "
                        f": {row['count']:>4,} "
                        f"({row['percentage']:>5.2f}%)"
                    )

            # =====================================================
            # 4️⃣ MESSENGER DICTIONARY
            # =====================================================
            if messenger_file:
                mtime = datetime.fromtimestamp(messenger_file.stat().st_mtime)
                print("\n💬 MESSENGER DICTIONARY")
                print("=" * 80)
                print(f"📄 File: {messenger_file.name}")
                print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")

                with open(messenger_file, "r", encoding="utf-8") as f:
                    m = json.load(f)

                print(f"   Total mentions: {m.get('total_mentions', 0):,}")
                print(f"   Messenger types: {m.get('total_messenger_types', 0)}\n")

                for row in m.get("messengers", []):
                    print(
                        f"   {row['messenger']:<10} "
                        f": {row['count']:>4,} "
                        f"({row['percentage']:>5.2f}%)"
                    )
        
        except Exception as e:
            print(f"❌ Error reading stats: {e}")

# 2. DICTIONARIES (NEWEST)
print("\n\n2️⃣ DICTIONARIES (NEWEST BATCH):")
print("-" * 80)

dict_dir = task_dir / "dictionaries"
if not dict_dir.exists():
    print(f"⚠️  Dictionaries directory not found: {dict_dir}")
else:
    dict_files = list(dict_dir.glob("*.csv"))

    if not dict_files:
        print("⚠️  No dictionary files found")
    else:
        # Sort newest first
        dict_files.sort(key=lambda f: f.stat().st_mtime, reverse=True)

        latest_time = dict_files[0].stat().st_mtime
        time_threshold = 10  # seconds
        latest_batch = [
            f for f in dict_files
            if abs(f.stat().st_mtime - latest_time) < time_threshold
        ]

        print(f"✓ Found {len(dict_files)} total dictionary file(s)")
        print(f"📄 Showing LATEST batch: {len(latest_batch)} file(s)\n")

        for dict_file in latest_batch:
            print("=" * 80)
            print(f"📄 File: {dict_file.name}")
            mtime = datetime.fromtimestamp(dict_file.stat().st_mtime)
            print(f"🕒 Modified: {mtime:%Y-%m-%d %H:%M:%S}")

            try:
                df = pd.read_csv(dict_file)

                # --------------------------------------------------
                # COUNTRY CODES DICTIONARY
                # --------------------------------------------------
                if set(df.columns) == {"country_code", "count", "percentage"}:
                    print("\n🌍 COUNTRY CODES DICTIONARY")
                    print(f"  Unique countries : {len(df)}")
                    print(f"  Total phones     : {df['count'].sum():,}\n")

                    df = df.sort_values("count", ascending=False)
                    print(df.head(10).to_string(index=False))

                # --------------------------------------------------
                # OPERATOR CODES DICTIONARY
                # --------------------------------------------------
                elif set(df.columns) == {"country_code", "operator_code", "count", "percentage"}:
                    print("\n📡 OPERATOR CODES DICTIONARY")
                    print(f"  Unique operators : {len(df)}")
                    print(f"  Total phones     : {df['count'].sum():,}\n")

                    df = df.sort_values("count", ascending=False)
                    print(df.head(10).to_string(index=False))

                # --------------------------------------------------
                # MESSENGER DICTIONARY
                # --------------------------------------------------
                elif set(df.columns) == {"messenger", "count", "percentage"}:
                    print("\n💬 MESSENGER MENTIONS DICTIONARY")
                    print(f"  Messenger types  : {len(df)}")
                    print(f"  Total mentions  : {df['count'].sum():,}\n")

                    df = df.sort_values("count", ascending=False)
                    print(df.to_string(index=False))

                # --------------------------------------------------
                # UNKNOWN / FUTURE DICTIONARY
                # --------------------------------------------------
                else:
                    print("\n❓ UNKNOWN DICTIONARY FORMAT")
                    print(f"  Columns: {list(df.columns)}\n")
                    print(df.head(10).to_string(index=False))

                print()

            except Exception as e:
                print(f"❌ Error reading {dict_file.name}: {e}")

# 3. VISUALIZATIONS (NEWEST BATCH)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                if 'country' in viz_file.name:
                    viz_name = 'Country Codes Distribution'
                elif 'operator' in viz_file.name:
                    viz_name = 'Operator Codes Distribution'
                elif 'messenger' in viz_file.name:
                    viz_name = 'Messenger Mentions Distribution'
                else:
                    viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**  
✅ Load data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure phone analysis with full parameters  
✅ Execute using operation.execute()  
✅ Analyze results and review artifacts  

**Key takeaways:**
- Phone analysis validates and parses phone numbers
- Country code extraction reveals geographic distribution
- Operator code analysis identifies carriers
- Messenger detection finds communication preferences
- All artifacts saved in structured directories

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_phone_analyzer_advanced.ipynb`** includes:
- Large dataset processing with Dask
- Parallel processing with Joblib
- Custom messenger pattern detection
- Country-specific operator analysis
- Phone normalization and validation
- Performance optimization

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)